[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/tracing/crewai_tracing_tutorial.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://raw.githubusercontent.com/Arize-ai/phoenix-assets/9e6101d95936f4bd4d390efc9ce646dc6937fb2d/images/socal/github-large-banner-phoenix.jpg" width="1000"/>
        <br>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>
<h1 align="center">使用 Arize Phoenix 追踪 CrewAI</h1>

In [ ]:
!pip install -q arize-phoenix opentelemetry-sdk opentelemetry-exporter-otlp crewai crewai_tools openinference-instrumentation-crewai

# 启动 Phoenix

In [ ]:
import phoenix as px

session = px.launch_app()

# 配置 OTel

In [ ]:
from phoenix.otel import register

tracer_provider = register(endpoint="http://localhost:6006/v1/traces")

# 为 CrewAI 添加埋点

In [ ]:
from openinference.instrumentation.crewai import CrewAIInstrumentor

CrewAIInstrumentor().instrument(skip_dep_check=True, tracer_provider=tracer_provider)

根据包版本不同，CrewAI 底层使用 LangChain 或 LiteLLM 调用 LLM。建议为对应库添加埋点，以观察 LLM 调用。

In [ ]:
! pip show crewai | grep Version

如果使用 CrewAI<0.63.0，请为 LangChain 添加埋点：

In [ ]:
# ! pip install openinference-instrumentation-langchain

# from openinference.instrumentation.langchain import LangChainInstrumentor

# LangChainInstrumentor().instrument(tracer_provider=tracer_provider)

如果使用 CrewAI>=0.63.0，请为 LiteLLM 添加埋点：

In [ ]:
! pip install openinference-instrumentation-litellm

from openinference.instrumentation.litellm import LiteLLMInstrumentor

LiteLLMInstrumentor().instrument(tracer_provider=tracer_provider)

# 密钥

注意：运行此 Colab 需要：

* OpenAI API 密钥 (https://openai.com/)
* Serper API 密钥 (https://serper.dev/)






In [ ]:
import getpass
import os

# 尚未设置 API 密钥时提示用户输入
openai_key = os.getenv("OPENAI_API_KEY", "OPENAI_API_KEY")
serper_key = os.getenv("SERPER_API_KEY", "SERPER_API_KEY")

if openai_key == "OPENAI_API_KEY":
    openai_key = getpass.getpass("请输入 OPENAI_API_KEY： ")

if serper_key == "SERPER_API_KEY":
    serper_key = getpass.getpass("请输入 SERPER_API_KEY： ")

# 使用提供的密钥设置环境变量
os.environ["OPENAI_API_KEY"] = openai_key
os.environ["SERPER_API_KEY"] = serper_key

In [ ]:
from crewai import Agent, Crew, Process, Task
from crewai_tools import SerperDevTool

search_tool = SerperDevTool()

# 定义智能体的角色与目标
researcher = Agent(
    role="资深研究分析师",
    goal="发掘 AI 与数据科学的前沿进展",
    backstory="""你在领先的技术智库工作。
  你擅长识别新兴趋势。
  你擅长剖析复杂数据并提出可操作的洞察。""",
    verbose=True,
    allow_delegation=False,
    # 可以通过可选 llm 属性指定使用的模型。
    # llm=ChatOpenAI(model_name="gpt-3.5", temperature=0.7),
    tools=[search_tool],
)
writer = Agent(
    role="技术内容策略师",
    goal="创作关于技术进展的精彩内容",
    backstory="""你是知名内容策略师，以富有洞察力、引人入胜的文章闻名。
  你将复杂概念转化为精彩叙述。""",
    verbose=True,
    allow_delegation=True,
)

# 为智能体创建任务
task1 = Task(
    description="""全面分析 2024 年 AI 的最新进展。
  识别关键趋势、突破性技术与潜在行业影响。""",
    expected_output="以要点列出的完整分析报告",
    agent=researcher,
)

task2 = Task(
    description="""根据提供的洞察，撰写引人入胜的博客
  文章，突出最重要的 AI 进展。
  文章应信息丰富且易于理解，面向熟悉技术的读者。
  表达生动，避免复杂词汇，让文章读起来自然。""",
    expected_output="至少 4 段的完整博客文章",
    agent=writer,
)

# 使用顺序流程实例化 Crew
crew = Crew(
    agents=[researcher, writer], tasks=[task1, task2], verbose=1, process=Process.sequential
)

# 启动 Crew！
result = crew.kickoff()

print("######################")
print(result)

# 查看结果

为方便使用，本指南整合为一个 Notebook。不过，也可以在 [Docker](https://arize.com/docs/phoenix/deployment/docker) 容器或本机中运行 Arize Phoenix。

In [ ]:
print(f"查看追踪： {session.url}")

 # ⭐⭐⭐ 如果本指南对你有帮助，请为 [Arize Phoenix](https://github.com/Arize-ai/phoenix) 点亮一颗星 ⭐⭐⭐